# Credit Pipeline (Cached Notebook) — Option 2

> هدف الدفتر: تحميل بيانات **prime** و **transactions** مرة واحدة في الذاكرة، وبعدها تقدر تعيد تشغيل باقي الخطوات بسرعة (Feature Engineering / Training / Evaluation) بدون انتظار الـ IO كل مرة.

> Rollback سهل: الدفتر جديد فقط + mapping قابل للتفعيل/الإيقاف من `config.ENABLE_MAPPING`.

## 1) Set Up Notebook Imports and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import importlib

# Ensure we import local modules from this folder (Jupyter doesn't define __file__)
try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

# Put notebook directory first so local .py modules win over any installed packages
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
else:
    # move to front
    sys.path.remove(NOTEBOOK_DIR)
    sys.path.insert(0, NOTEBOOK_DIR)

# Debug: show what folder we're importing from
print("Notebook dir:", NOTEBOOK_DIR)
print("sys.path[0]:", sys.path[0])

import config

# If a wrong module got imported earlier in the kernel, reload from disk
import feature_engineering as _fe
_fe_path = getattr(_fe, "__file__", "")
if (not _fe_path) or (os.path.abspath(_fe_path) != os.path.abspath(os.path.join(NOTEBOOK_DIR, "feature_engineering.py"))):
    print("[warning] feature_engineering imported from:", _fe_path)
    print("Reloading feature_engineering from local file...")
    _fe = importlib.reload(_fe)

from data_loader import load_prime_data, load_transaction_data, merge_data
from feature_engineering import engineer_prime_features, engineer_transaction_features, create_target
from preprocessing import preprocess
from model import train_xgboost, save_model, load_model, get_top_features_by_gain, subset_to_features
from evaluation import evaluate, generate_report, find_best_threshold_fbeta
from pipeline import check_for_leakage  # reuse leakage check

try:
    from mapping_integration import apply_mapping_layer
except Exception:
    apply_mapping_layer = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
np.random.seed(getattr(config, "RANDOM_STATE", 42))

print("Notebook ready")
print("ENABLE_MAPPING:", getattr(config, "ENABLE_MAPPING", False))

## 2) Load Dataset (Prime + Transactions) **once**
- هذه الخلية هي اللي بتاخد وقت. شغّلها مرة واحدة، وبعدها اشتغل على بقية الخلايا براحتك.
- لو غيّرت ملفات الداتا على الديسك، ارجع شغّل هذه الخلية بس.

In [ ]:
# Load once (cached in memory)
prime_df = load_prime_data()
txn_df   = load_transaction_data()

print("prime_df:", prime_df.shape)
print("txn_df:", txn_df.shape)
display(prime_df.head(3))
display(txn_df.head(3))

## 3) (Optional) Mapping Layer (from `Credit-code/mapping/`)
- لو `config.ENABLE_MAPPING = True` هنضيف CUSTOMER_ID ونستخدمه لتجميع المعاملات بدل RIMNO.
- لو مش محتاج mapping، سيبه False وهتفضل كل حاجة زي ما هي.

In [ ]:
mapped_id_col = None
if getattr(config, "ENABLE_MAPPING", False):
    if apply_mapping_layer is None:
        raise RuntimeError("mapping_integration.py not importable. Ensure it exists.")
    prime_df_m, txn_df_m, mapped_id_col, mapping_stats = apply_mapping_layer(
        prime_df,
        txn_df,
        mapped_id_col=getattr(config, "MAPPED_CUSTOMER_ID_COL", "CUSTOMER_ID"),
    )
    print(mapping_stats)
else:
    prime_df_m, txn_df_m = prime_df.copy(), txn_df.copy()
    print("Mapping disabled")

print("mapped_id_col:", mapped_id_col)

## 4) Feature Engineering + Merge
- تقدر تعيد تشغيل هذه الخلية عدة مرات بسرعة بدون إعادة تحميل الداتا.

In [ ]:
prime_fe = engineer_prime_features(prime_df_m)

# If mapping enabled, aggregate txn features by mapped id
if mapped_id_col:
    old_cid = config.CUSTOMER_ID
    config.CUSTOMER_ID = mapped_id_col
    try:
        txn_fe = engineer_transaction_features(txn_df_m)
    finally:
        config.CUSTOMER_ID = old_cid
else:
    txn_fe = engineer_transaction_features(txn_df_m)

merged = merge_data(prime_fe, txn_fe)
print("merged:", merged.shape)
display(merged.head(3))

## 5) Create Target (and optional sampling)

In [ ]:
merged_t = create_target(merged)
target = merged_t[config.TARGET_COL]

sample_weight = merged_t["sample_weight"] if "sample_weight" in merged_t.columns else None

print("target value counts:")
display(target.value_counts(dropna=False))

# Optional sampling (same behavior as CLI)
USE_SAMPLE = False  # set True for quick iteration
if USE_SAMPLE:
    from sklearn.model_selection import train_test_split
    keep_idx, _ = train_test_split(
        merged_t.index,
        train_size=0.25,
        stratify=target,
        random_state=config.RANDOM_STATE,
    )
    merged_t = merged_t.loc[keep_idx]
    target = target.loc[keep_idx]
    if sample_weight is not None:
        sample_weight = sample_weight.loc[keep_idx]
    print("Sampling enabled ->", merged_t.shape)

## 6) Preprocess Data (fit)

In [ ]:
X, y, artifacts = preprocess(merged_t, target, fit=True)
if sample_weight is not None:
    sample_weight = sample_weight.loc[X.index]

print("X:", X.shape, " y:", y.shape)

## 7) Leakage Check

In [ ]:
suspects = check_for_leakage(X, y, abort_on_leak=True)
print("Leakage suspects:", suspects[:10])

## 8) Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

split_kwargs = dict(
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_STATE,
 )

if sample_weight is not None:
    X_train, X_test, y_train, y_test, sw_train, sw_test = train_test_split(
        X, y, sample_weight, **split_kwargs
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, **split_kwargs)
    sw_train = None
    sw_test = None

print("X_train:", X_train.shape, "X_test:", X_test.shape)

## 9) Train Model (Option 2: XGBoost) + Find Best Threshold

In [ ]:
import xgboost as xgb

bst = train_xgboost(
    X_train, y_train,
    X_test, y_test,
    sample_weight_train=sw_train,
 )

dtest = xgb.DMatrix(X_test, feature_names=list(X_test.columns))
y_proba = bst.predict(dtest)
best_threshold = find_best_threshold_fbeta(y_test, y_proba, beta=2.0)
y_pred = (y_proba >= best_threshold).astype(int)

print("best_threshold:", best_threshold)

## 10) Hyperparameter Tuning (optional)
(Skipped here — use `tune_hyperparameters()` if you want CV tuning.)

## 11) Evaluate Model (Top-Gain Feature Selection + Retrain)

In [ ]:
top_n = getattr(config, "TOP_N_GAIN_FEATURES", 20)
selected_features = get_top_features_by_gain(
    bst,
    artifacts.get("feature_order", list(X_train.columns)),
    top_n=top_n,
)
print("Selected features:", len(selected_features))
display(pd.Series(selected_features).head(10))

X_train2, X_test2, X_all2, artifacts2 = subset_to_features(
    X_train, X_test, X, artifacts, selected_features
)

bst2 = train_xgboost(
    X_train2, y_train,
    X_test2, y_test,
    sample_weight_train=sw_train,
 )

dtest2 = xgb.DMatrix(X_test2, feature_names=list(X_test2.columns))
y_proba2 = bst2.predict(dtest2)
best_threshold2 = find_best_threshold_fbeta(y_test, y_proba2, beta=2.0)
y_pred2 = (y_proba2 >= best_threshold2).astype(int)

print("best_threshold2:", best_threshold2)

## 12) Save/Load Model Artifacts + Scores

In [ ]:
metrics = evaluate(y_test, y_pred2, y_proba2, threshold=best_threshold2)
print(metrics)

_ = generate_report(metrics, y_test, y_pred2, config.REPORT_PATH)
print("Report saved:", config.REPORT_PATH)

# Score all rows using the final model on the selected features
dall = xgb.DMatrix(X_all2, feature_names=list(X_all2.columns))
all_proba = bst2.predict(dall)
all_pred = (all_proba >= best_threshold2).astype(int)

# Choose id column in output
id_col_out = mapped_id_col if mapped_id_col else config.CUSTOMER_ID
if id_col_out not in merged_t.columns:
    id_col_out = config.CUSTOMER_ID
customer_ids_out = merged_t.loc[X_all2.index, id_col_out]

scores_df = pd.DataFrame({
    id_col_out: customer_ids_out.values,
    "default_probability": all_proba,
    "predicted_label": all_pred,
})
scores_df.to_csv(config.SCORES_PATH, index=False)
print("Scores saved:", config.SCORES_PATH)
display(scores_df.head(5))

# Save model + artifacts + threshold + selected features
save_model(
    bst2,
    {**artifacts2, "best_threshold": best_threshold2, "selected_features": selected_features, "id_col": id_col_out},
)
print("Model saved:", config.MODEL_PATH)

# Reload and quick inference demo
model_loaded, art_loaded = load_model(config.MODEL_PATH)
sample_rows = merged_t.sample(n=min(50, len(merged_t)), random_state=config.RANDOM_STATE)
X_new, _, _ = preprocess(sample_rows, y=None, fit=False, artifacts=art_loaded)
sel = art_loaded.get("selected_features")
if sel:
    X_new = X_new[sel]
dnew = xgb.DMatrix(X_new, feature_names=list(X_new.columns))
proba_new = model_loaded.predict(dnew)
thr = art_loaded.get("best_threshold", 0.5)
pred_new = (proba_new >= thr).astype(int)

out = sample_rows[[id_col_out]].copy() if id_col_out in sample_rows.columns else sample_rows.copy()
out["default_probability"] = proba_new
out["predicted_label"] = pred_new
display(out.head(10))